In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np

# ==========================================
# 1. DATASET DEFINITION
# ==========================================
class ASDImageDataset(Dataset):
    def __init__(self, autistic_dir, non_autistic_dir, transform=None):
        self.filepaths = []
        self.labels = []
        self.transform = transform
        
        # Load Autistic Images (Label 1)
        for img_name in os.listdir(autistic_dir):
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                self.filepaths.append(os.path.join(autistic_dir, img_name))
                self.labels.append(1)
                
        # Load Non-Autistic Images (Label 0)
        for img_name in os.listdir(non_autistic_dir):
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                self.filepaths.append(os.path.join(non_autistic_dir, img_name))
                self.labels.append(0)

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img_path = self.filepaths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.float32)

# ==========================================
# 2. PREPROCESSING & AUGMENTATION
# ==========================================
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ==========================================
# 3. DATA SPLITTING & LOADERS
# ==========================================
autistic_path = '/kaggle/input/datasets/radhgarg/autism-image-dataset/autism_images/autistic'
non_autistic_path = '/kaggle/input/datasets/radhgarg/autism-image-dataset/autism_images/non_autistic'

# Load full dataset (using train transforms initially, we will handle val/test safely)
full_dataset = ASDImageDataset(autistic_path, non_autistic_path, transform=train_transforms)

total_size = len(full_dataset)
train_size = int(0.8 * total_size)
val_size = int(0.1 * total_size)
test_size = total_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size], 
    generator=torch.Generator().manual_seed(42) # For reproducibility
)

# Overwrite transforms for validation and testing to remove random augmentations
val_dataset.dataset.transform = val_test_transforms
test_dataset.dataset.transform = val_test_transforms

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

print(f"Dataset Split: {train_size} Train | {val_size} Validation | {test_size} Test")

# ==========================================
# 4. MODEL INITIALIZATION (DUAL T4 GPU)
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 1) # Binary classification

if torch.cuda.device_count() > 1:
    print(f"Utilizing {torch.cuda.device_count()} GPUs via DataParallel.")
    model = nn.DataParallel(model)

model = model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# ==========================================
# 5. TRAINING & VALIDATION LOOP
# ==========================================
num_epochs = 10

print("\n--- Starting Training ---")
for epoch in range(num_epochs):
    # Training Phase
    model.train()
    train_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        labels = labels.unsqueeze(1)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * inputs.size(0)
        
    train_loss = train_loss / len(train_loader.dataset)
    
    # Validation Phase
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            labels = labels.unsqueeze(1)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            
    val_loss = val_loss / len(val_loader.dataset)
    
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

# ==========================================
# 6. TESTING & EVALUATION
# ==========================================
print("\n--- Starting Final Evaluation on Unseen Test Data ---")
model.eval()
all_preds = []
all_true = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        outputs = model(inputs)
        probs = torch.sigmoid(outputs)
        preds = torch.round(probs)
        
        all_preds.extend(preds.cpu().numpy())
        all_true.extend(labels.cpu().numpy())

all_preds = np.array(all_preds).flatten()
all_true = np.array(all_true).flatten()

# Calculate Metrics
accuracy = accuracy_score(all_true, all_preds)
precision = precision_score(all_true, all_preds, zero_division=0)
recall = recall_score(all_true, all_preds, zero_division=0)
f1 = f1_score(all_true, all_preds, zero_division=0)
conf_matrix = confusion_matrix(all_true, all_preds)

print(f"Test Accuracy:  {accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall:    {recall:.4f}")
print(f"Test F1-Score:  {f1:.4f}")
print("\nConfusion Matrix:")
print(conf_matrix)
print("(Row 1: True Non-Autistic | Row 2: True Autistic)")

Dataset Split: 2348 Train | 293 Validation | 295 Test
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 158MB/s] 


Utilizing 2 GPUs via DataParallel.

--- Starting Training ---
Epoch 1/10 | Train Loss: 0.4411 | Val Loss: 0.3809
Epoch 2/10 | Train Loss: 0.1346 | Val Loss: 0.4054
Epoch 3/10 | Train Loss: 0.0377 | Val Loss: 0.4429
Epoch 4/10 | Train Loss: 0.0144 | Val Loss: 0.4372
Epoch 5/10 | Train Loss: 0.0132 | Val Loss: 0.4878
Epoch 6/10 | Train Loss: 0.0121 | Val Loss: 0.5178
Epoch 7/10 | Train Loss: 0.0068 | Val Loss: 0.4814
Epoch 8/10 | Train Loss: 0.0189 | Val Loss: 0.6417
Epoch 9/10 | Train Loss: 0.0397 | Val Loss: 0.7059
Epoch 10/10 | Train Loss: 0.0716 | Val Loss: 0.6744

--- Starting Final Evaluation on Unseen Test Data ---
Test Accuracy:  0.8339
Test Precision: 0.7907
Test Recall:    0.9128
Test F1-Score:  0.8474

Confusion Matrix:
[[110  36]
 [ 13 136]]
(Row 1: True Non-Autistic | Row 2: True Autistic)


# with data generation

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# ==========================================
# 1. HYPERPARAMETERS & SETUP
# ==========================================
torch.manual_seed(42) # For reproducible research results

BATCH_SIZE = 64  
EPOCHS = 15
LEARNING_RATE = 0.0001
WEIGHT_DECAY = 1e-4  

DATA_DIR = '/kaggle/input/datasets/radhgarg/autism-image-dataset/autism_images'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==========================================
# 2. DATA AUGMENTATION & LOADING
# ==========================================
# Training Transforms (Fights overfitting)
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Evaluation Transforms (Strictly standardized)
eval_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root=DATA_DIR)

train_size = int(0.8 * len(full_dataset))
val_size = int(0.1 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size
train_data, val_data, test_data = random_split(full_dataset, [train_size, val_size, test_size])

class DatasetWrapper(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
        
    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y
        
    def __len__(self):
        return len(self.subset)

train_dataset = DatasetWrapper(train_data, transform=train_transforms)
val_dataset = DatasetWrapper(val_data, transform=eval_transforms)
test_dataset = DatasetWrapper(test_data, transform=eval_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Dataset Split: {len(train_dataset)} Train | {len(val_dataset)} Validation | {len(test_dataset)} Test")

# ==========================================
# 3. MODEL ARCHITECTURE
# ==========================================
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

for param in model.parameters():
    param.requires_grad = False

for param in model.layer4.parameters():
    param.requires_grad = True

num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(p=0.5),                  
    nn.Linear(num_ftrs, 256),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(256, 1)                   
)

if torch.cuda.device_count() > 1:
    print(f"Utilizing {torch.cuda.device_count()} GPUs via DataParallel.")
    model = nn.DataParallel(model)

model = model.to(device)

# ==========================================
# 4. TRAINING LOOP
# ==========================================
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), 
                       lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

print("\n--- Starting Training ---")
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.float().unsqueeze(1).to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * inputs.size(0)
        
    train_loss /= len(train_loader.dataset)
    
    model.eval()
    val_loss = 0.0
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.float().unsqueeze(1).to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            
    val_loss /= len(val_loader.dataset)
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_resnet_model.pth')

# ==========================================
# 5. FINAL EVALUATION ON UNSEEN TEST DATA
# ==========================================
print("\n--- Starting Final Evaluation on Unseen Test Data ---")

# Load the best weights saved during training
model.load_state_dict(torch.load('best_resnet_model.pth'))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.float().unsqueeze(1).to(device)
        outputs = model(inputs)
        
        # Apply sigmoid to convert logits to probabilities, then threshold at 0.5
        probs = torch.sigmoid(outputs)
        preds = (probs >= 0.5).float()
        
        # Move back to CPU for sklearn metrics
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Convert lists to flat numpy arrays
all_preds = np.array(all_preds).flatten()
all_labels = np.array(all_labels).flatten()

# Calculate Metrics
acc = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds)
rec = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)
conf_matrix = confusion_matrix(all_labels, all_preds)

print(f"Test Accuracy:  {acc:.4f}")
print(f"Test Precision: {prec:.4f}")
print(f"Test Recall:    {rec:.4f}")
print(f"Test F1-Score:  {f1:.4f}\n")
print("Confusion Matrix:")
print(conf_matrix)
print("(Row 1: True Non-Autistic | Row 2: True Autistic)")

Using device: cuda
Dataset Split: 2348 Train | 293 Validation | 295 Test
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 191MB/s]


Utilizing 2 GPUs via DataParallel.

--- Starting Training ---
Epoch 1/15 | Train Loss: 0.5347 | Val Loss: 0.4252
Epoch 2/15 | Train Loss: 0.3908 | Val Loss: 0.4052
Epoch 3/15 | Train Loss: 0.3452 | Val Loss: 0.4075
Epoch 4/15 | Train Loss: 0.2887 | Val Loss: 0.4628
Epoch 5/15 | Train Loss: 0.2410 | Val Loss: 0.5829
Epoch 6/15 | Train Loss: 0.2193 | Val Loss: 0.5216
Epoch 7/15 | Train Loss: 0.2107 | Val Loss: 0.5702
Epoch 8/15 | Train Loss: 0.1806 | Val Loss: 0.6011
Epoch 9/15 | Train Loss: 0.1592 | Val Loss: 0.6719
Epoch 10/15 | Train Loss: 0.1621 | Val Loss: 0.6031
Epoch 11/15 | Train Loss: 0.1437 | Val Loss: 0.6419
Epoch 12/15 | Train Loss: 0.1186 | Val Loss: 0.5697
Epoch 13/15 | Train Loss: 0.1133 | Val Loss: 0.7682
Epoch 14/15 | Train Loss: 0.1139 | Val Loss: 0.6924
Epoch 15/15 | Train Loss: 0.0952 | Val Loss: 0.6615

--- Starting Final Evaluation on Unseen Test Data ---
Test Accuracy:  0.8542
Test Precision: 0.8323
Test Recall:    0.8836
Test F1-Score:  0.8571

Confusion Matrix:
[

In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# ==========================================
# 1. HYPERPARAMETERS & SETUP
# ==========================================
torch.manual_seed(42)

BATCH_SIZE = 64  
EPOCHS = 20           
LEARNING_RATE = 1e-4  
WEIGHT_DECAY = 1e-3   

DATA_DIR = '/kaggle/input/datasets/radhgarg/autism-image-dataset/autism_images'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 2. DATA AUGMENTATION
# ==========================================
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root=DATA_DIR)
train_size = int(0.8 * len(full_dataset))
val_size = int(0.1 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size
train_data, val_data, test_data = random_split(full_dataset, [train_size, val_size, test_size])

class DatasetWrapper(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform: x = self.transform(x)
        return x, y
    def __len__(self): return len(self.subset)

train_loader = DataLoader(DatasetWrapper(train_data, train_transforms), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(DatasetWrapper(val_data, eval_transforms), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(DatasetWrapper(test_data, eval_transforms), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ==========================================
# 3. ADVANCED ARCHITECTURE 
# ==========================================
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

for param in model.parameters():
    param.requires_grad = False

for param in model.layer3.parameters():
    param.requires_grad = True
for param in model.layer4.parameters():
    param.requires_grad = True

num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(p=0.5),                  
    nn.Linear(num_ftrs, 256),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(256, 1)                   
)

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

model = model.to(device)

# ==========================================
# 4. ADAMW, SCHEDULER & LOSS
# ==========================================
criterion = nn.BCEWithLogitsLoss()

base_model = model.module if isinstance(model, nn.DataParallel) else model

optimizer = optim.AdamW([
    {'params': base_model.layer3.parameters(), 'lr': LEARNING_RATE / 10}, 
    {'params': base_model.layer4.parameters(), 'lr': LEARNING_RATE},      
    {'params': base_model.fc.parameters(), 'lr': LEARNING_RATE}           
], weight_decay=WEIGHT_DECAY)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

# ==========================================
# 5. TRAINING LOOP WITH EARLY STOPPING
# ==========================================
print("\n--- Starting Advanced Training ---")
best_val_loss = float('inf')
early_stop_counter = 0
patience = 5 

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.float().unsqueeze(1).to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * inputs.size(0)
        
    train_loss /= len(train_loader.dataset)
    
    model.eval()
    val_loss = 0.0
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.float().unsqueeze(1).to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            
    val_loss /= len(val_loader.dataset)
    
    scheduler.step(val_loss)
    
    current_lr = optimizer.param_groups[1]['lr']
    print(f"Epoch {epoch+1}/{EPOCHS} | LR: {current_lr:.6f} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_resnet_model.pth')
        early_stop_counter = 0
    else:
        early_stop_counter += 1
        if early_stop_counter >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}. Model hasn't improved in {patience} epochs.")
            break

# ==========================================
# 6. FINAL EVALUATION
# ==========================================
print("\n--- Starting Final Evaluation on Unseen Test Data ---")
model.load_state_dict(torch.load('best_resnet_model.pth'))
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.float().unsqueeze(1).to(device)
        outputs = model(inputs)
        probs = torch.sigmoid(outputs)
        preds = (probs >= 0.5).float()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds).flatten()
all_labels = np.array(all_labels).flatten()

print(f"Test Accuracy:  {accuracy_score(all_labels, all_preds):.4f}")
print(f"Test Precision: {precision_score(all_labels, all_preds):.4f}")
print(f"Test Recall:    {recall_score(all_labels, all_preds):.4f}")
print(f"Test F1-Score:  {f1_score(all_labels, all_preds):.4f}\n")
print("Confusion Matrix:\n", confusion_matrix(all_labels, all_preds))


--- Starting Advanced Training ---
Epoch 1/20 | LR: 0.000100 | Train Loss: 0.5323 | Val Loss: 0.4272
Epoch 2/20 | LR: 0.000100 | Train Loss: 0.3858 | Val Loss: 0.3992
Epoch 3/20 | LR: 0.000100 | Train Loss: 0.3379 | Val Loss: 0.4023
Epoch 4/20 | LR: 0.000100 | Train Loss: 0.2755 | Val Loss: 0.4646
Epoch 5/20 | LR: 0.000050 | Train Loss: 0.2255 | Val Loss: 0.6130
Epoch 6/20 | LR: 0.000050 | Train Loss: 0.1802 | Val Loss: 0.4958
Epoch 7/20 | LR: 0.000050 | Train Loss: 0.1605 | Val Loss: 0.5351
Early stopping triggered at epoch 7. Model hasn't improved in 5 epochs.

--- Starting Final Evaluation on Unseen Test Data ---
Test Accuracy:  0.8712
Test Precision: 0.8699
Test Recall:    0.8699
Test F1-Score:  0.8699

Confusion Matrix:
 [[130  19]
 [ 19 127]]


# resnet replaced with efficientnet

In [2]:
# import os
# import torch
# import torch.nn as nn
# import torch.optim as optim
# import torch.nn.functional as F
# import numpy as np
# from torchvision import datasets, transforms, models
# from torch.utils.data import DataLoader, random_split
# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# # ==========================================
# # 1. HYPERPARAMETERS & HARDWARE SETUP
# # ==========================================
# torch.manual_seed(42)

# # FIXED: Force deterministic cuDNN algorithms to prevent Kaggle T4 execution crashes
# torch.backends.cudnn.benchmark = False
# torch.backends.cudnn.deterministic = True

# BATCH_SIZE = 32       
# EPOCHS = 25           
# LEARNING_RATE = 1e-4  
# WEIGHT_DECAY = 1e-3   

# DATA_DIR = '/kaggle/input/datasets/radhgarg/autism-image-dataset/autism_images'
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # ==========================================
# # 2. CUSTOM FOCAL LOSS FOR HARD EXAMPLES
# # ==========================================
# class FocalLoss(nn.Module):
#     def __init__(self, alpha=0.25, gamma=2.0):
#         super(FocalLoss, self).__init__()
#         self.alpha = alpha
#         self.gamma = gamma

#     def forward(self, inputs, targets):
#         BCE_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
#         pt = torch.exp(-BCE_loss) 
#         F_loss = self.alpha * (1-pt)**self.gamma * BCE_loss
#         return torch.mean(F_loss)

# # ==========================================
# # 3. DATA AUGMENTATION
# # ==========================================
# train_transforms = transforms.Compose([
#     transforms.Resize((256, 256)),
#     transforms.RandomHorizontalFlip(p=0.5),
#     transforms.RandomRotation(degrees=15),
#     transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
#     transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# ])

# eval_transforms = transforms.Compose([
#     transforms.Resize((256, 256)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# ])

# full_dataset = datasets.ImageFolder(root=DATA_DIR)
# train_size = int(0.8 * len(full_dataset))
# val_size = int(0.1 * len(full_dataset))
# test_size = len(full_dataset) - train_size - val_size
# train_data, val_data, test_data = random_split(full_dataset, [train_size, val_size, test_size])

# class DatasetWrapper(torch.utils.data.Dataset):
#     def __init__(self, subset, transform=None):
#         self.subset = subset
#         self.transform = transform
#     def __getitem__(self, index):
#         x, y = self.subset[index]
#         if self.transform: x = self.transform(x)
#         return x, y
#     def __len__(self): return len(self.subset)

# train_loader = DataLoader(DatasetWrapper(train_data, train_transforms), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
# val_loader = DataLoader(DatasetWrapper(val_data, eval_transforms), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
# test_loader = DataLoader(DatasetWrapper(test_data, eval_transforms), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# # ==========================================
# # 4. EFFICIENTNET-B3 ARCHITECTURE
# # ==========================================
# model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)

# for param in model.features.parameters():
#     param.requires_grad = False

# for param in model.features[6].parameters():
#     param.requires_grad = True
# for param in model.features[7].parameters():
#     param.requires_grad = True

# in_features = model.classifier[1].in_features
# model.classifier = nn.Sequential(
#     nn.Dropout(p=0.4, inplace=True),
#     nn.Linear(in_features, 256),
#     nn.BatchNorm1d(256), 
#     nn.SiLU(),           
#     nn.Dropout(p=0.3, inplace=True),
#     nn.Linear(256, 1)
# )

# if torch.cuda.device_count() > 1:
#     model = nn.DataParallel(model)

# model = model.to(device)

# # ==========================================
# # 5. OPTIMIZER, SCHEDULER & FOCAL LOSS
# # ==========================================
# criterion = FocalLoss(alpha=0.25, gamma=2.0)

# base_model = model.module if isinstance(model, nn.DataParallel) else model

# optimizer = optim.AdamW([
#     {'params': base_model.features[6].parameters(), 'lr': LEARNING_RATE / 10}, 
#     {'params': base_model.features[7].parameters(), 'lr': LEARNING_RATE / 5},      
#     {'params': base_model.classifier.parameters(), 'lr': LEARNING_RATE}           
# ], weight_decay=WEIGHT_DECAY)

# scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

# # ==========================================
# # 6. TRAINING LOOP
# # ==========================================
# print("\n--- Starting EfficientNet + Focal Loss Training ---")
# best_val_loss = float('inf')
# early_stop_counter = 0
# patience = 6 

# for epoch in range(EPOCHS):
#     model.train()
#     train_loss = 0.0
    
#     for inputs, labels in train_loader:
#         inputs, labels = inputs.to(device), labels.float().unsqueeze(1).to(device)
#         optimizer.zero_grad()
#         outputs = model(inputs)
#         loss = criterion(outputs, labels)
#         loss.backward()
#         optimizer.step()
#         train_loss += loss.item() * inputs.size(0)
        
#     train_loss /= len(train_loader.dataset)
    
#     model.eval()
#     val_loss = 0.0
    
#     with torch.no_grad():
#         for inputs, labels in val_loader:
#             inputs, labels = inputs.to(device), labels.float().unsqueeze(1).to(device)
#             outputs = model(inputs)
#             loss = criterion(outputs, labels)
#             val_loss += loss.item() * inputs.size(0)
            
#     val_loss /= len(val_loader.dataset)
    
#     scheduler.step(val_loss)
    
#     current_lr = optimizer.param_groups[2]['lr']
#     print(f"Epoch {epoch+1}/{EPOCHS} | Head LR: {current_lr:.6f} | Train Focal Loss: {train_loss:.4f} | Val Focal Loss: {val_loss:.4f}")
    
#     if val_loss < best_val_loss:
#         best_val_loss = val_loss
#         torch.save(model.state_dict(), 'best_effnet_model.pth')
#         early_stop_counter = 0
#     else:
#         early_stop_counter += 1
#         if early_stop_counter >= patience:
#             print(f"Early stopping triggered at epoch {epoch+1}.")
#             break

# # ==========================================
# # 7. FINAL EVALUATION
# # ==========================================
# print("\n--- Final Evaluation (Unseen Test Data) ---")
# model.load_state_dict(torch.load('best_effnet_model.pth'))
# model.eval()

# all_preds, all_labels = [], []

# with torch.no_grad():
#     for inputs, labels in test_loader:
#         inputs, labels = inputs.to(device), labels.float().unsqueeze(1).to(device)
#         outputs = model(inputs)
#         probs = torch.sigmoid(outputs)
#         preds = (probs >= 0.5).float()
#         all_preds.extend(preds.cpu().numpy())
#         all_labels.extend(labels.cpu().numpy())

# all_preds = np.array(all_preds).flatten()
# all_labels = np.array(all_labels).flatten()

# print(f"Test Accuracy:  {accuracy_score(all_labels, all_preds):.4f}")
# print(f"Test Precision: {precision_score(all_labels, all_preds):.4f}")
# print(f"Test Recall:    {recall_score(all_labels, all_preds):.4f}")
# print(f"Test F1-Score:  {f1_score(all_labels, all_preds):.4f}\n")
# print("Confusion Matrix:\n", confusion_matrix(all_labels, all_preds))

AcceleratorError: CUDA error: misaligned address
Search for `cudaErrorMisalignedAddress' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
